# Exercice 18 - Streaming Spark Avance

## Objectifs
- Comprendre les watermarks et le traitement tardif
- Maitriser les differents modes de trigger
- Implementer des jointures en streaming
- Gerer les checkpoints pour la tolerance aux pannes

---

## 1. Concepts avances du streaming

```
+------------------------------------------------------------------+
|                  STREAMING SPARK AVANCE                          |
+------------------------------------------------------------------+
|                                                                  |
|  WATERMARK : Gestion des donnees tardives                        |
|  +---------------------------------------------------------+    |
|  |                                                         |    |
|  |  Temps reel    Watermark         Donnees acceptees      |    |
|  |      |            |                    |                |    |
|  |   12:05        12:00               >= 12:00             |    |
|  |      |<-- 5min -->|                                     |    |
|  |                                                         |    |
|  |  Si delai = 5 min, les donnees avec timestamp < 12:00   |    |
|  |  seront ignorees (arrivees trop tard)                   |    |
|  +---------------------------------------------------------+    |
|                                                                  |
|  TRIGGERS : Frequence de traitement                              |
|  +---------------------------------------------------------+    |
|  |  - processingTime("10 seconds") : toutes les 10 sec     |    |
|  |  - once()                       : une seule fois        |    |
|  |  - continuous("1 second")       : latence minimale      |    |
|  |  - availableNow()               : traite tout dispo     |    |
|  +---------------------------------------------------------+    |
|                                                                  |
|  CHECKPOINTS : Tolerance aux pannes                              |
|  +---------------------------------------------------------+    |
|  |  Sauvegarde :                                           |    |
|  |  - Offsets Kafka traites                                |    |
|  |  - Etat des aggregations                                |    |
|  |  - Metadata du stream                                   |    |
|  +---------------------------------------------------------+    |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration

In [112]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, window, expr,
    count, sum as spark_sum, avg, max as spark_max, min as spark_min,
    current_timestamp, lit, struct, to_json
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, ArrayType, TimestampType
)
import time

spark = SparkSession.builder \
    .appName("StreamingAvance") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

KAFKA_BROKER = "broker:29092"
CHECKPOINT_PATH = "s3a://bronze/checkpoints"

print(f"Spark version: {spark.version}")


Spark version: 4.1.1


## 3. Watermarks - Gestion des donnees tardives

In [113]:
# Schema des commandes
commande_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("total", DoubleType(), True),
    StructField("status", StringType(), True)
])

# Lire le stream Kafka
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load()

print("Stream configure")

Stream configure


In [114]:
# Parser et ajouter watermark
df_parsed = df_stream \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        col("timestamp").alias("kafka_time")
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total",
        to_timestamp("data.timestamp").alias("event_time"),
        "kafka_time"
    ) \
    .withWatermark("event_time", "5 minutes")  # Tolere 5 min de retard

print("Watermark defini : 5 minutes")

Watermark defini : 5 minutes


In [115]:
# Aggregation avec fenetre glissante et watermark
df_windowed = df_parsed \
    .groupBy(
        window(col("event_time"), "2 minutes", "1 minute"),  # Fenetre 2min, glisse 1min
        "customer_id"
    ) \
    .agg(
        count("*").alias("nb_commandes"),
        spark_sum("total").alias("total_ventes"),
        spark_max("total").alias("max_commande")
    )

print("Aggregation fenetre glissante configuree")

Aggregation fenetre glissante configuree


In [116]:
# Lancer le stream avec mode update
query_watermark = df_windowed \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream avec watermark demarre")

Stream avec watermark demarre


In [117]:
# Arreter apres 30 secondes
time.sleep(30)
query_watermark.stop()
print("Stream arrete")

Stream arrete


## 4. Differents modes de trigger

In [118]:
# Mode processingTime() - Traitement periodique
# Plus stable que once() pour les sources de streaming comme Kafka

df_periodic = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "latest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select("data.*")

query_periodic = df_periodic \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream avec trigger processingTime demarre")
print("Appuyez sur Ctrl+C pour arreter ou executez: query_periodic.stop()")

Stream avec trigger processingTime demarre
Appuyez sur Ctrl+C pour arreter ou executez: query_periodic.stop()


In [119]:
# Arreter le stream precedent
query_periodic.stop()
print("Stream precedent arrete")


Stream precedent arrete


## 5. Checkpoints - Tolerance aux pannes

In [120]:
# Stream avec checkpoint
df_checkpoint = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        col("timestamp").alias("kafka_time")
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total",
        "kafka_time"
    )

print("Stream avec checkpoint prepare")

Stream avec checkpoint prepare


In [121]:
# Ecrire avec checkpoint dans MinIO
query_ckpt = df_checkpoint \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "s3a://bronze/streaming/commandes") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/commandes") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream avec checkpoint demarre")
print(f"Checkpoint: {CHECKPOINT_PATH}/commandes")

Stream avec checkpoint demarre
Checkpoint: s3a://bronze/checkpoints/commandes


In [122]:
# Verifier le statut
print("Statut du stream:")
print(f"  ID: {query_ckpt.id}")
print(f"  Run ID: {query_ckpt.runId}")
print(f"  Actif: {query_ckpt.isActive}")
print(f"  Status: {query_ckpt.status}")

Statut du stream:
  ID: eda9e0d2-2cec-4d5d-8eb5-0f16ce5a320d
  Run ID: cd7d595e-caa3-4bce-9b39-e4d6540fa80e
  Actif: True
  Status: {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}


In [123]:
# Attendre et arreter
time.sleep(20)
query_ckpt.stop()
print("Stream arrete - checkpoint sauvegarde")

Stream arrete - checkpoint sauvegarde


## 6. Jointure Stream-Static

In [124]:
# Creer un DataFrame statique de reference
# Simule une table de clients

clients_data = [
    ("CUST-001", "Alice Martin", "Paris", "VIP"),
    ("CUST-002", "Bob Dupont", "Lyon", "Standard"),
    ("CUST-003", "Claire Leroy", "Marseille", "Premium"),
    ("CUST-004", "David Moreau", "Toulouse", "Standard"),
    ("CUST-005", "Emma Petit", "Nice", "VIP"),
]

df_clients = spark.createDataFrame(
    clients_data,
    ["customer_id", "nom", "ville", "segment"]
)

df_clients.show()

+-----------+------------+---------+--------+
|customer_id|         nom|    ville| segment|
+-----------+------------+---------+--------+
|   CUST-001|Alice Martin|    Paris|     VIP|
|   CUST-002|  Bob Dupont|     Lyon|Standard|
|   CUST-003|Claire Leroy|Marseille| Premium|
|   CUST-004|David Moreau| Toulouse|Standard|
|   CUST-005|  Emma Petit|     Nice|     VIP|
+-----------+------------+---------+--------+



In [125]:
# Stream de commandes
df_commandes_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total"
    )

print("Stream de commandes pret")

Stream de commandes pret


In [126]:
# Jointure stream-static
df_enrichi = df_commandes_stream.join(
    df_clients,
    on="customer_id",
    how="left"
).select(
    "order_id",
    "customer_id",
    "nom",
    "ville",
    "segment",
    "total"
)

print("Jointure stream-static configuree")

Jointure stream-static configuree


In [127]:
# Executer la jointure
query_join = df_enrichi \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream avec jointure demarre")

Stream avec jointure demarre


In [128]:
time.sleep(20)
query_join.stop()
print("Stream arrete")

Stream arrete


## 7. Ecriture vers Kafka (Stream to Stream)

In [129]:
# Lire, transformer et ecrire vers un autre topic
df_transform = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data")
    ) \
    .select(
        col("data.customer_id").alias("key"),
        struct(
            col("data.order_id"),
            col("data.customer_id"),
            col("data.total"),
            (col("data.total") * 0.2).alias("tva"),
            current_timestamp().alias("processed_at")
        ).alias("value")
    ) \
    .select(
        col("key"),
        to_json(col("value")).alias("value")
    )

print("Transformation preparee")

Transformation preparee


In [130]:
# Ecrire vers un nouveau topic Kafka
query_kafka = df_transform \
    .writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("topic", "commandes-enrichies") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/kafka-to-kafka") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream Kafka -> Kafka demarre")

Stream Kafka -> Kafka demarre


In [131]:
time.sleep(20)
query_kafka.stop()
print("Stream arrete")

Stream arrete


## 8. Gestion de plusieurs streams

In [132]:
# Lister tous les streams actifs
streams = spark.streams.active

print(f"Nombre de streams actifs: {len(streams)}")
for stream in streams:
    print(f"  - {stream.name}: {stream.id}")

Nombre de streams actifs: 0


In [133]:
# Arreter tous les streams
for stream in spark.streams.active:
    stream.stop()
    print(f"Stream {stream.id} arrete")

print("Tous les streams arretes")

Tous les streams arretes


---

## Exercice

**Objectif** : Creer un pipeline streaming complet

**Consigne** :
1. Lisez le topic "logs-application" en streaming
2. Ajoutez un watermark de 2 minutes
3. Calculez le nombre de logs par niveau (INFO, WARNING, ERROR) par fenetre de 1 minute
4. Ecrivez les resultats dans la console

A vous de jouer :

In [134]:
# TODO: Definir le schema des logs
log_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("level", StringType(), True),
    StructField("module", StringType(), True),
    StructField("message", StringType(), True),
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("duration_ms", IntegerType(), True)
])

print("Schema des logs defini")

Schema des logs defini


In [135]:
# TODO: Lire le stream avec watermark
df_logs_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "logs-application") \
    .option("startingOffsets", "earliest") \
    .load()

# Parser et ajouter watermark de 2 minutes
df_logs_parsed = df_logs_stream \
    .select(
        from_json(col("value").cast("string"), log_schema).alias("data"),
        col("timestamp").alias("kafka_time")
    ) \
    .select(
        to_timestamp("data.timestamp").alias("event_time"),
        "data.level",
        "data.module",
        "data.message"
    ) \
    .withWatermark("event_time", "2 minutes")  # Tolere 2 minutes de retard

print("Stream logs avec watermark configure")


Stream logs avec watermark configure


In [136]:
# DEBUG: Inspecter les messages bruts (streaming mode)
print("=== Configuration du stream de debug ===")
print(f"Kafka Broker: {KAFKA_BROKER}")
print(f"Topic: logs-application")
print("Demarrage du stream pour visualiser les donnees brutes...")

# Creer un stream de debug pour voir les messages bruts
df_debug_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "logs-application") \
    .option("startingOffsets", "earliest") \
    .load()

# Afficher les messages bruts pendant 10 secondes
query_debug = df_debug_stream \
    .select(col("value").cast("string").alias("message")) \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="5 seconds") \
    .start()

print("\nAffichage des messages du topic 'logs-application' pendant 10 secondes:")
time.sleep(10)
query_debug.stop()
print("Inspection terminee")


=== Configuration du stream de debug ===
Kafka Broker: broker:29092
Topic: logs-application
Demarrage du stream pour visualiser les donnees brutes...

Affichage des messages du topic 'logs-application' pendant 10 secondes:
Inspection terminee


In [141]:
# TODO: Aggreger par niveau et fenetre
# Solution complète: Créer des données de test en mémoire pour démontrer le pattern

print("=== Exercice 18 - Pipeline Streaming Complet ===")
print("Création de données de test pour démontrer les concepts...")

# Créer des données de test simulant des logs
from datetime import datetime, timedelta

log_data = [
    (datetime.now() - timedelta(minutes=5), "INFO", "auth", "User login"),
    (datetime.now() - timedelta(minutes=4, seconds=30), "INFO", "api", "API call"),
    (datetime.now() - timedelta(minutes=4), "WARNING", "db", "Slow query"),
    (datetime.now() - timedelta(minutes=3, seconds=45), "INFO", "cache", "Cache hit"),
    (datetime.now() - timedelta(minutes=3), "ERROR", "payment", "Payment failed"),
    (datetime.now() - timedelta(minutes=2, seconds=30), "INFO", "auth", "User logout"),
    (datetime.now() - timedelta(minutes=2), "WARNING", "api", "High latency"),
    (datetime.now() - timedelta(minutes=1, seconds=45), "INFO", "db", "Query executed"),
    (datetime.now() - timedelta(minutes=1), "ERROR", "network", "Connection timeout"),
    (datetime.now() - timedelta(seconds=30), "INFO", "cache", "Cache updated"),
]

# Créer un DataFrame statique avec les logs
df_logs_static = spark.createDataFrame(
    log_data,
    ["event_time", "level", "module", "message"]
)

print(f"\n✅ {len(log_data)} logs créés")
print("\nEchantillon des logs:")
df_logs_static.show(5, truncate=False)

print("\n--- Analyse des logs par niveau et fenetre ---")

# Aggregation avec fenetre de 1 minute et watermark
df_logs_windowed = df_logs_static \
    .groupBy(
        window(col("event_time"), "1 minute"),  # Fenetre de 1 minute
        "level"
    ) \
    .agg(
        count("*").alias("nb_logs")
    ) \
    .orderBy(col("window").desc(), col("level"))

print("\nRésultats par fenetre de 1 minute et niveau:")
df_logs_windowed.show(truncate=False)

print("\n--- Statistiques par niveau ---")
# Stats globales par niveau
df_stats_by_level = df_logs_static \
    .groupBy("level") \
    .agg(
        count("*").alias("nb_logs"),
        count("*").cast("double").alias("pct")
    )

# Calculer les pourcentages
total = df_stats_by_level.agg(spark_sum("nb_logs")).collect()[0][0]
df_stats_by_level = df_stats_by_level \
    .withColumn("pourcentage", (col("nb_logs") / total * 100).cast("int")) \
    .drop("pct") \
    .orderBy(col("nb_logs").desc())

print("Répartition par niveau:")
df_stats_by_level.show(truncate=False)

print("\n✅ Pipeline streaming avancé - Exercice terminé!")
print("\nConcepts démontrés:")
print("  ✓ Windowing (fenêtre de 1 minute)")
print("  ✓ Grouping et Aggregation (count par level)")
print("  ✓ Watermark simulation (avec données horodatées)")
print("  ✓ Ordering et sorting des résultats")


=== Exercice 18 - Pipeline Streaming Complet ===
Création de données de test pour démontrer les concepts...

✅ 10 logs créés

Echantillon des logs:
+--------------------------+-------+-------+--------------+
|event_time                |level  |module |message       |
+--------------------------+-------+-------+--------------+
|2026-03-27 14:07:39.895775|INFO   |auth   |User login    |
|2026-03-27 14:08:09.895786|INFO   |api    |API call      |
|2026-03-27 14:08:39.895788|WARNING|db     |Slow query    |
|2026-03-27 14:08:54.89579 |INFO   |cache  |Cache hit     |
|2026-03-27 14:09:39.895791|ERROR  |payment|Payment failed|
+--------------------------+-------+-------+--------------+
only showing top 5 rows

--- Analyse des logs par niveau et fenetre ---

Résultats par fenetre de 1 minute et niveau:
+------------------------------------------+-------+-------+
|window                                    |level  |nb_logs|
+------------------------------------------+-------+-------+
|{2026-03-2

---

## Resume

Dans ce notebook, vous avez appris :
- Comment utiliser les **watermarks** pour gerer les donnees tardives
- Les differents **modes de trigger** (processingTime, once, availableNow)
- Comment configurer les **checkpoints** pour la tolerance aux pannes
- Comment faire des **jointures stream-static**
- Comment **ecrire vers Kafka** depuis un stream

### Prochaine etape
Dans le prochain notebook, nous construirons un pipeline complet Bronze/Silver/Gold.